In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, copy, pickle
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# GMDCSA arrays saved earlier
d = pickle.load(open('/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24_arrays.pkl', 'rb'))
X, y, subj = d['X'], d['y'], d['subj']
print(X.shape, y.shape, sorted(set(subj)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
(160, 100, 6) (160,) [np.str_('Subject 1'), np.str_('Subject 2'), np.str_('Subject 3'), np.str_('Subject 4')]


In [3]:
import pickle
from sklearn.model_selection import train_test_split

with open('/content/drive/MyDrive/J26-DS-320/datasets/urfd_features.pkl', 'rb') as f:
    urfd_seqs = pickle.load(f)

FEATURE_NAMES = ["nose_y", "shoulder_y", "hip_y", "aspect_ratio", "width", "height"]
T = 100

def resample_by_index(feats, T=T):
    arr = np.array([[f[n] for n in FEATURE_NAMES] for f in feats], dtype=np.float64)
    grid = np.linspace(0, len(arr) - 1, T)
    return np.stack([np.interp(grid, np.arange(len(arr)), arr[:, j]) for j in range(arr.shape[1])], axis=1).astype(np.float32)

Xu = np.array([resample_by_index(s['features']) for s in urfd_seqs])
yu = np.array([s['label'] for s in urfd_seqs])
nu = [s['name'] for s in urfd_seqs]

# EXACT original split (same parameters as your notebook, same order of sequences)
idx = np.arange(len(yu))
i_tr, i_tmp, _, y_tmp = train_test_split(idx, yu, test_size=0.3, stratify=yu, random_state=42)
i_va, i_te = train_test_split(i_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

Xu_tr, yu_tr = Xu[i_tr], yu[i_tr]
Xu_va, yu_va = Xu[i_va], yu[i_va]
Xu_te, yu_te = Xu[i_te], yu[i_te]
print('URFD train/val/test:', len(i_tr), len(i_va), len(i_te))
print('test names:', [nu[i] for i in i_te])

URFD train/val/test: 49 10 11
test names: ['adl-11-cam0-rgb', 'adl-29-cam0-rgb', 'adl-15-cam0-rgb', 'fall-28-cam0-rgb', 'adl-33-cam0-rgb', 'fall-07-cam0-rgb', 'fall-04-cam0-rgb', 'adl-23-cam0-rgb', 'adl-36-cam0-rgb', 'fall-14-cam0-rgb', 'fall-18-cam0-rgb']


In [5]:
class FallCNNLSTM(nn.Module):
    def __init__(self, num_features=6, cnn_channels=[32, 64, 128], lstm_hidden=128, lstm_layers=2):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, cnn_channels[0], kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cnn_channels[0], cnn_channels[1], kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(cnn_channels[1], cnn_channels[2], kernel_size=3, padding=1)
        self.relu = nn.ReLU(); self.dropout = nn.Dropout(0.3)
        self.lstm = nn.LSTM(cnn_channels[2], lstm_hidden, num_layers=lstm_layers, batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(lstm_hidden, 64); self.fc2 = nn.Linear(64, 2)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.dropout(self.relu(self.conv1(x)))
        x = self.dropout(self.relu(self.conv2(x)))
        x = self.dropout(self.relu(self.conv3(x)))
        _, (h_n, _) = self.lstm(x.permute(0, 2, 1))
        return self.fc2(self.relu(self.fc1(h_n[-1])))

def loader(Xa, ya, shuffle):
    return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya, dtype=torch.long)), batch_size=8, shuffle=shuffle)

def predict(model, Xa):
    model.eval()
    with torch.no_grad():
        return torch.argmax(model(torch.tensor(Xa).to(device)), dim=1).cpu().numpy()

def train_model(Xtr, ytr, Xva, yva, epochs=50, patience=10):
    model = FallCNNLSTM().to(device)
    w = torch.tensor([1.0 / c for c in np.bincount(ytr)], dtype=torch.float32).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for bx, by in loader(Xtr, ytr, True):
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad(); crit(model(bx), by).backward(); opt.step()
        f1 = f1_score(yva, predict(model, Xva), zero_division=0)
        if f1 > best_f1: best_f1, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model

def metrics(yt, yp):
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {"precision": float(precision_score(yt, yp, zero_division=0)),
            "recall": float(recall_score(yt, yp, zero_division=0)),
            "f1": float(f1_score(yt, yp, zero_division=0)),
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

In [6]:
# ---------- C1: leave-one-GMDCSA-subject-out, URFD (train+val only) always in training ----------
torch.manual_seed(42); np.random.seed(42)
u_X = np.concatenate([Xu_tr, Xu_va]); u_y = np.concatenate([yu_tr, yu_va])

c1, t_all, p_all = {}, [], []
for held in sorted(set(subj)):
    te = subj == held
    gX, gy = X[~te], y[~te]
    gX_tr, gX_va, gy_tr, gy_va = train_test_split(gX, gy, test_size=0.15, stratify=gy, random_state=42)
    Xtr = np.concatenate([u_X, gX_tr]); ytr = np.concatenate([u_y, gy_tr])
    m = train_model(Xtr, ytr, gX_va, gy_va)
    yp = predict(m, X[te])
    c1[held] = metrics(y[te], yp)
    t_all += list(y[te]); p_all += list(yp)
    print(held, c1[held])
c1_pooled = metrics(np.array(t_all), np.array(p_all))
print('\nC1 POOLED (URFD + GMDCSA):', c1_pooled)
print('GMDCSA-only pooled was: f1=0.901  precision=0.837  recall=0.975')

# ---------- C2: final combined model, tested on the original URFD test set ----------
torch.manual_seed(42); np.random.seed(42)
gX_tr, gX_va, gy_tr, gy_va = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
Xtr = np.concatenate([Xu_tr, gX_tr]); ytr = np.concatenate([yu_tr, gy_tr])
Xva = np.concatenate([Xu_va, gX_va]); yva = np.concatenate([yu_va, gy_va])
final = train_model(Xtr, ytr, Xva, yva)
c2 = metrics(yu_te, predict(final, Xu_te))
print('\nC2 combined model on original URFD test (11 seq):', c2, '| original URFD-only model: f1=0.889')

torch.save(final.state_dict(), '/content/drive/MyDrive/J26-DS-320/datasets/fall_cnn_lstm_v2.pt')
json.dump({"c1_loso_with_urfd_per_subject": c1, "c1_pooled": c1_pooled,
           "gmdcsa_only_pooled_f1": 0.9006, "c2_urfd_test": c2, "urfd_only_f1": 0.889},
          open('/content/drive/MyDrive/J26-DS-320/datasets/combined_model_results.json', 'w'), indent=2)

Subject 1 {'precision': 0.8421052631578947, 'recall': 1.0, 'f1': 0.9142857142857143, 'tp': 16, 'fp': 3, 'fn': 0, 'tn': 13}
Subject 2 {'precision': 0.8064516129032258, 'recall': 1.0, 'f1': 0.8928571428571429, 'tp': 25, 'fp': 6, 'fn': 0, 'tn': 17}
Subject 3 {'precision': 0.9130434782608695, 'recall': 1.0, 'f1': 0.9545454545454546, 'tp': 21, 'fp': 2, 'fn': 0, 'tn': 20}
Subject 4 {'precision': 0.7142857142857143, 'recall': 0.8823529411764706, 'f1': 0.7894736842105263, 'tp': 15, 'fp': 6, 'fn': 2, 'tn': 14}

C1 POOLED (URFD + GMDCSA): {'precision': 0.8191489361702128, 'recall': 0.9746835443037974, 'f1': 0.8901734104046243, 'tp': 77, 'fp': 17, 'fn': 2, 'tn': 64}
GMDCSA-only pooled was: f1=0.901  precision=0.837  recall=0.975

C2 combined model on original URFD test (11 seq): {'precision': 0.625, 'recall': 1.0, 'f1': 0.7692307692307693, 'tp': 5, 'fp': 3, 'fn': 0, 'tn': 3} | original URFD-only model: f1=0.889


In [8]:
  from google.colab import files
  files.download('/content/drive/MyDrive/J26-DS-320/datasets/combined_model_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>